In [1]:
import os
os.getcwd()
# os.chdir(path)    # or you can set your working dir.

'/Users/xingfuxu/PycharmProjects/EquityPremiumPrediction3'

In [2]:
# Your working dir should include "Perform_CW_test.py" and "NN_models.py" files so that the following functions can work
from Perform_CW_test import CW_test
from NN_models import Net3

In [3]:
## In-sample Model Building
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.linear_model import LinearRegression
from sklearn.linear_model import LassoCV, ElasticNetCV
from sklearn.cross_decomposition import PLSRegression
from sklearn.decomposition import PCA
from Perform_Selection_IC import select_IC
from scipy.stats import zscore
import torch

#
from sklearn.svm import SVR
from sklearn.neighbors import KNeighborsRegressor
from sklearn.ensemble import AdaBoostRegressor
from xgboost import XGBRegressor
#
from sklearn.metrics import mean_squared_error
import matplotlib.pyplot as plt

In [4]:
predictor_df = pd.read_csv('result_predictor.csv')
predictor_df.head()

,month,log_equity_premium,equity_premium,DP,DY,EP,SVAR,BM,NTIS,TBL,...,MA_2_9,MA_2_12,MA_3_9,MA_3_12,MOM_1,MOM_2,MOM_3,MOM_6,MOM_9,MOM_12
0,192701,-0.005710,-0.00571,-2.942374,-2.963349,-2.374773,0.00047,0.44371,0.05082,3.23,...,1,1,1,1,0,0,1,1,1,1
1,192702,0.042017,0.04302,-2.979535,-2.932946,-2.430353,0.00029,0.42850,0.05167,3.29,...,1,1,1,1,1,1,1,1,1,1
2,192703,0.004697,0.00472,-2.976535,-2.970053,-2.445079,0.00092,0.46977,0.04636,3.20,...,1,1,1,1,1,1,1,1,1,1
3,192704,0.009940,0.01002,-2.984225,-2.967143,-2.471309,0.00060,0.45675,0.05051,3.39,...,1,1,1,1,1,1,1,1,1,1
4,192705,0.057987,0.05985,-3.025963,-2.975058,-2.531446,0.00039,0.43478,0.05528,3.33,...,1,1,1,1,1,1,1,1,1,1


In [5]:
# remove irrelavent columns
predictor0 = predictor_df.drop(['month', 'equity_premium'], axis=1)

# get all the predictors and set the log equity premium 1-month ahead
predictor = np.concatenate([predictor0['log_equity_premium'][1:].values.reshape(-1, 1),
                            predictor0.iloc[0:(predictor0.shape[0] - 1), 1:]], axis=1)
# number of rows
N = predictor.shape[0]

# number of all columns, including the log equity premium
n_cols = predictor.shape[1]

# Actual one-month ahead log equity premium
actual = predictor[:, [0]]

# Historical average forecast as benchmark
y_pred_HA = predictor0['log_equity_premium'].values[0:(predictor0.shape[0] - 1), ].cumsum() / np.arange(1, N + 1)
y_pred_HA = y_pred_HA.reshape(-1, 1)

In [6]:
## Machine learning methods used in GKX (2020)
# OLS, or the Kitchen sink model
OLS = LinearRegression()
OLS.fit(predictor[:, 1:], predictor[:, [0]])
y_pred_OLS = OLS.predict(predictor[:, 1:]).reshape(-1, 1)

# PLS
PLS2 = PLSRegression()   # n_components=2 is the default setting
PLS2.fit(predictor[:, 1:], predictor[:, [0]])
y_pred_PLS = PLS2.predict(predictor[:, 1:]).reshape(-1, 1)

# PCR
k_max_variables = 3
X_train = predictor[:, 1:]
pca = PCA()
pca.fit(zscore(X_train, axis=0, ddof=1))
X_train_new = pca.transform(zscore(X_train, axis=0, ddof=1))
k_components = select_IC(predictor[:, [0]], X_train_new[:, :k_max_variables], IC=3)
F_hat_selected = X_train_new[:, :k_components]
PCR = LinearRegression()
PCR.fit(F_hat_selected, predictor[:, [0]])
y_pred_PCR = PCR.predict(F_hat_selected).reshape(-1, 1)

# LASSO
LASSO = LassoCV(cv=10)
LASSO.fit(predictor[:, 1:], predictor[:, 0])
y_pred_LASSO = LASSO.predict(predictor[:, 1:]).reshape(-1, 1)

# Elastic Net (ENet)
ENet = ElasticNetCV(cv=10)
ENet.fit(predictor[:, 1:], predictor[:, 0])
y_pred_ENet = ENet.predict(predictor[:, 1:]).reshape(-1, 1)


# Gradient boosted regression trees (GBRT)
GBRT = GradientBoostingRegressor()
GBRT.fit(predictor[:, 1:], predictor[:, 0])
y_pred_GBRT = GBRT.predict(predictor[:, 1:]).reshape(-1, 1)


# Random Forest (RF)
RF = RandomForestRegressor(random_state=0)
RF.fit(predictor[:, 1:], predictor[:, 0])
y_pred_RF = RF.predict(predictor[:, 1:]).reshape(-1, 1)


# Neural Networks with SGD
torch.manual_seed(1)
np.random.seed(1)
NN3 = Net3(n_cols - 1, 32, 16, 8, 1)
optimizer = torch.optim.SGD(NN3.parameters(), lr=0.1)
loss_func = torch.nn.MSELoss()
X_train_tensor = torch.tensor(predictor[:, 1:], dtype=torch.float)
y_train_tensor = torch.tensor(predictor[:, [0]], dtype=torch.float)

losses = []
for i in range(1000):
    out = NN3(X_train_tensor)
    loss = loss_func(out, y_train_tensor)
    optimizer.zero_grad()
    losses.append(loss.item())
    loss.backward()
    optimizer.step()

losses = np.array(losses)
y_pred_NN3 = NN3(X_train_tensor).detach().numpy()


## Other commonly used machine learning method
# Support Vector Regression (SVR)
SVR_model = SVR()
SVR_model.fit(predictor[:, 1:], predictor[:, 0])
y_pred_SVR = SVR_model.predict(predictor[:, 1:]).reshape(-1, 1)
# K Neighbors Regressor (KNR)
KNR = KNeighborsRegressor()
KNR.fit(predictor[:, 1:], predictor[:, 0])
y_pred_KNR = KNR.predict(predictor[:, 1:]).reshape(-1, 1)
# AdaBoost
AdaBoost = AdaBoostRegressor()
AdaBoost.fit(predictor[:, 1:], predictor[:, 0])
y_pred_AdaBoost = AdaBoost.predict(predictor[:, 1:]).reshape(-1, 1)
# XGBoost
XGBoost = XGBRegressor()
XGBoost.fit(predictor[:, 1:], predictor[:, 0])
y_pred_XGBoost = XGBoost.predict(predictor[:, 1:]).reshape(-1, 1)
# Combination
y_pred_combination = np.concatenate([y_pred_OLS, y_pred_PLS, y_pred_PCR, y_pred_LASSO, y_pred_ENet, 
                                     y_pred_GBRT, y_pred_RF, y_pred_NN3, y_pred_SVR, y_pred_KNR, 
                                     y_pred_AdaBoost, y_pred_XGBoost], axis=1).mean(axis=1).reshape(-1, 1)

In [7]:
## In-sample prediction results for all years (ALL)
# OLS
MSFE_HA = mean_squared_error(y_pred_HA, actual)
MSFE_OLS_ALL = mean_squared_error(y_pred_OLS, actual)
IS_R_OLS_ALL = 1 - MSFE_OLS_ALL / MSFE_HA
MSFE_adjusted_OLS_ALL, p_OLS_ALL = CW_test(actual, y_pred_HA, y_pred_OLS)
# PLS
MSFE_PLS_ALL = mean_squared_error(y_pred_PLS, actual)
IS_R_PLS_ALL = 1 - MSFE_PLS_ALL / MSFE_HA
MSFE_adjusted_PLS_ALL, p_PLS_ALL = CW_test(actual, y_pred_HA, y_pred_PLS)
# PCR
MSFE_PCR_ALL = mean_squared_error(y_pred_PCR, actual)
IS_R_PCR_ALL = 1 - MSFE_PCR_ALL / MSFE_HA
MSFE_adjusted_PCR_ALL, p_PCR_ALL = CW_test(actual, y_pred_HA, y_pred_PCR)
# ENet
MSFE_ENet_ALL = mean_squared_error(y_pred_ENet, actual)
IS_R_ENet_ALL = 1 - MSFE_ENet_ALL / MSFE_HA
MSFE_adjusted_ENet_ALL, p_ENet_ALL = CW_test(actual, y_pred_HA, y_pred_ENet)
# LASSO
MSFE_LASSO_ALL = mean_squared_error(y_pred_LASSO, actual)
IS_R_LASSO_ALL = 1 - MSFE_LASSO_ALL / MSFE_HA
MSFE_adjusted_LASSO_ALL, p_LASSO_ALL = CW_test(actual, y_pred_HA, y_pred_LASSO)
# GBRT
MSFE_GBRT_ALL = mean_squared_error(y_pred_GBRT, actual)
IS_R_GBRT_ALL = 1 - MSFE_GBRT_ALL / MSFE_HA
MSFE_adjusted_GBRT_ALL, p_GBRT_ALL = CW_test(actual, y_pred_HA, y_pred_GBRT)
# RF
MSFE_RF_ALL = mean_squared_error(y_pred_RF, actual)
IS_R_RF_ALL = 1 - MSFE_RF_ALL / MSFE_HA
MSFE_adjusted_RF_ALL, p_RF_ALL = CW_test(actual, y_pred_HA, y_pred_RF)
# NN3
MSFE_NN3_ALL = mean_squared_error(y_pred_NN3, actual)
IS_R_NN3_ALL = 1 - MSFE_NN3_ALL / MSFE_HA
MSFE_adjusted_NN3_ALL, p_NN3_ALL = CW_test(actual, y_pred_HA, y_pred_NN3)
# Other commonly used ML methods
# SVR
MSFE_SVR_ALL = mean_squared_error(y_pred_SVR, actual)
IS_R_SVR_ALL = 1 - MSFE_SVR_ALL / MSFE_HA
MSFE_adjusted_SVR_ALL, p_SVR_ALL = CW_test(actual, y_pred_HA, y_pred_SVR)
# KNR
MSFE_KNR_ALL = mean_squared_error(y_pred_KNR, actual)
IS_R_KNR_ALL = 1 - MSFE_KNR_ALL / MSFE_HA
MSFE_adjusted_KNR_ALL, p_KNR_ALL = CW_test(actual, y_pred_HA, y_pred_KNR)
# AdaBoost
MSFE_AdaBoost_ALL = mean_squared_error(y_pred_AdaBoost, actual)
IS_R_AdaBoost_ALL = 1 - MSFE_AdaBoost_ALL / MSFE_HA
MSFE_adjusted_AdaBoost_ALL, p_AdaBoost_ALL = CW_test(actual, y_pred_HA, y_pred_AdaBoost)
# XGBoost
MSFE_XGBoost_ALL = mean_squared_error(y_pred_XGBoost, actual)
IS_R_XGBoost_ALL = 1 - MSFE_XGBoost_ALL / MSFE_HA
MSFE_adjusted_XGBoost_ALL, p_XGBoost_ALL = CW_test(actual, y_pred_HA, y_pred_XGBoost)
# Combination
MSFE_combination_ALL = mean_squared_error(y_pred_combination, actual)
IS_R_combination_ALL = 1 - MSFE_combination_ALL / MSFE_HA
MSFE_adjusted_combination_ALL, p_combination_ALL = CW_test(actual, y_pred_HA, y_pred_combination)


In [8]:
## Prediction results for forecasts beginning at 1957:01
in_out_1957 = predictor_df.index[predictor_df['month'] == 195701][0]
# OLS
MSFE_HA = mean_squared_error(y_pred_HA[in_out_1957:, ], actual[in_out_1957:, ])
MSFE_OLS_1957 = mean_squared_error(y_pred_OLS[in_out_1957:, ], actual[in_out_1957:, ])
IS_R_OLS_1957 = 1 - MSFE_OLS_1957 / MSFE_HA
MSFE_adjusted_OLS_1957, p_OLS_1957 = CW_test(actual[in_out_1957:, ], y_pred_HA[in_out_1957:, ], y_pred_OLS[in_out_1957:, ])
# PLS
MSFE_PLS_1957 = mean_squared_error(y_pred_PLS[in_out_1957:, ], actual[in_out_1957:, ])
IS_R_PLS_1957 = 1 - MSFE_PLS_1957 / MSFE_HA
MSFE_adjusted_PLS_1957, p_PLS_1957 = CW_test(actual[in_out_1957:, ], y_pred_HA[in_out_1957:, ], y_pred_PLS[in_out_1957:, ])
# PCR
MSFE_PCR_1957 = mean_squared_error(y_pred_PCR[in_out_1957:, ], actual[in_out_1957:, ])
IS_R_PCR_1957 = 1 - MSFE_PCR_1957 / MSFE_HA
MSFE_adjusted_PCR_1957, p_PCR_1957 = CW_test(actual[in_out_1957:, ], y_pred_HA[in_out_1957:, ], y_pred_PCR[in_out_1957:, ])
# ENet
MSFE_ENet_1957 = mean_squared_error(y_pred_ENet[in_out_1957:, ], actual[in_out_1957:, ])
IS_R_ENet_1957 = 1 - MSFE_ENet_1957 / MSFE_HA
MSFE_adjusted_ENet_1957, p_ENet_1957 = CW_test(actual[in_out_1957:, ], y_pred_HA[in_out_1957:, ], y_pred_ENet[in_out_1957:, ])
# LASSO
MSFE_LASSO_1957 = mean_squared_error(y_pred_LASSO[in_out_1957:, ], actual[in_out_1957:, ])
IS_R_LASSO_1957 = 1 - MSFE_LASSO_1957 / MSFE_HA
MSFE_adjusted_LASSO_1957, p_LASSO_1957 = CW_test(actual[in_out_1957:, ], y_pred_HA[in_out_1957:, ], y_pred_LASSO[in_out_1957:, ])
# GBRT
MSFE_GBRT_1957 = mean_squared_error(y_pred_GBRT[in_out_1957:, ], actual[in_out_1957:, ])
IS_R_GBRT_1957 = 1 - MSFE_GBRT_1957 / MSFE_HA
MSFE_adjusted_GBRT_1957, p_GBRT_1957 = CW_test(actual[in_out_1957:, ], y_pred_HA[in_out_1957:, ], y_pred_GBRT[in_out_1957:, ])
# RF
MSFE_RF_1957 = mean_squared_error(y_pred_RF[in_out_1957:, ], actual[in_out_1957:, ])
IS_R_RF_1957 = 1 - MSFE_RF_1957 / MSFE_HA
MSFE_adjusted_RF_1957, p_RF_1957 = CW_test(actual[in_out_1957:, ], y_pred_HA[in_out_1957:, ], y_pred_RF[in_out_1957:, ])
# NN3
MSFE_NN3_1957 = mean_squared_error(y_pred_NN3[in_out_1957:, ], actual[in_out_1957:, ])
IS_R_NN3_1957 = 1 - MSFE_NN3_1957 / MSFE_HA
MSFE_adjusted_NN3_1957, p_NN3_1957 = CW_test(actual[in_out_1957:, ], y_pred_HA[in_out_1957:, ], y_pred_NN3[in_out_1957:, ])
## Other commonly used ML methods
# SVR
MSFE_SVR_1957 = mean_squared_error(y_pred_SVR[in_out_1957:, ], actual[in_out_1957:, ])
IS_R_SVR_1957 = 1 - MSFE_SVR_1957 / MSFE_HA
MSFE_adjusted_SVR_1957, p_SVR_1957 = CW_test(actual[in_out_1957:, ], y_pred_HA[in_out_1957:, ], y_pred_SVR[in_out_1957:, ])
# KNR
MSFE_KNR_1957 = mean_squared_error(y_pred_KNR[in_out_1957:, ], actual[in_out_1957:, ])
IS_R_KNR_1957 = 1 - MSFE_KNR_1957 / MSFE_HA
MSFE_adjusted_KNR_1957, p_KNR_1957 = CW_test(actual[in_out_1957:, ], y_pred_HA[in_out_1957:, ], y_pred_KNR[in_out_1957:, ])
# AdaBoost
MSFE_AdaBoost_1957 = mean_squared_error(y_pred_AdaBoost[in_out_1957:, ], actual[in_out_1957:, ])
IS_R_AdaBoost_1957 = 1 - MSFE_AdaBoost_1957 / MSFE_HA
MSFE_adjusted_AdaBoost_1957, p_AdaBoost_1957 = CW_test(actual[in_out_1957:, ], y_pred_HA[in_out_1957:, ], y_pred_AdaBoost[in_out_1957:, ])
# XGBoost
MSFE_XGBoost_1957 = mean_squared_error(y_pred_XGBoost[in_out_1957:, ], actual[in_out_1957:, ])
IS_R_XGBoost_1957 = 1 - MSFE_XGBoost_1957 / MSFE_HA
MSFE_adjusted_XGBoost_1957, p_XGBoost_1957 = CW_test(actual[in_out_1957:, ], y_pred_HA[in_out_1957:, ], y_pred_XGBoost[in_out_1957:, ])
# Combination
MSFE_combination_1957 = mean_squared_error(y_pred_combination[in_out_1957:, ], actual[in_out_1957:, ])
IS_R_combination_1957 = 1 - MSFE_combination_1957 / MSFE_HA
MSFE_adjusted_combination_1957, p_combination_1957 = CW_test(actual[in_out_1957:, ], y_pred_HA[in_out_1957:, ], y_pred_combination[in_out_1957:, ])

In [9]:
## Prediction results for forecasts beginning at 1990:01
in_out_1990 = predictor_df.index[predictor_df['month'] == 199001][0]
# OLS
MSFE_HA = mean_squared_error(y_pred_HA[in_out_1990:, ], actual[in_out_1990:, ])
MSFE_OLS_1990 = mean_squared_error(y_pred_OLS[in_out_1990:, ], actual[in_out_1990:, ])
IS_R_OLS_1990 = 1 - MSFE_OLS_1990 / MSFE_HA
MSFE_adjusted_OLS_1990, p_OLS_1990 = CW_test(actual[in_out_1990:, ], y_pred_HA[in_out_1990:, ], y_pred_OLS[in_out_1990:, ])
# PLS
MSFE_PLS_1990 = mean_squared_error(y_pred_PLS[in_out_1990:, ], actual[in_out_1990:, ])
IS_R_PLS_1990 = 1 - MSFE_PLS_1990 / MSFE_HA
MSFE_adjusted_PLS_1990, p_PLS_1990 = CW_test(actual[in_out_1990:, ], y_pred_HA[in_out_1990:, ], y_pred_PLS[in_out_1990:, ])
# PCR
MSFE_PCR_1990 = mean_squared_error(y_pred_PCR[in_out_1990:, ], actual[in_out_1990:, ])
IS_R_PCR_1990 = 1 - MSFE_PCR_1990 / MSFE_HA
MSFE_adjusted_PCR_1990, p_PCR_1990 = CW_test(actual[in_out_1990:, ], y_pred_HA[in_out_1990:, ], y_pred_PCR[in_out_1990:, ])
# ENet
MSFE_ENet_1990 = mean_squared_error(y_pred_ENet[in_out_1990:, ], actual[in_out_1990:, ])
IS_R_ENet_1990 = 1 - MSFE_ENet_1990 / MSFE_HA
MSFE_adjusted_ENet_1990, p_ENet_1990 = CW_test(actual[in_out_1990:, ], y_pred_HA[in_out_1990:, ], y_pred_ENet[in_out_1990:, ])
# LASSO
MSFE_LASSO_1990 = mean_squared_error(y_pred_LASSO[in_out_1990:, ], actual[in_out_1990:, ])
IS_R_LASSO_1990 = 1 - MSFE_LASSO_1990 / MSFE_HA
MSFE_adjusted_LASSO_1990, p_LASSO_1990 = CW_test(actual[in_out_1990:, ], y_pred_HA[in_out_1990:, ], y_pred_LASSO[in_out_1990:, ])
# GBRT
MSFE_GBRT_1990 = mean_squared_error(y_pred_GBRT[in_out_1990:, ], actual[in_out_1990:, ])
IS_R_GBRT_1990 = 1 - MSFE_GBRT_1990 / MSFE_HA
MSFE_adjusted_GBRT_1990, p_GBRT_1990 = CW_test(actual[in_out_1990:, ], y_pred_HA[in_out_1990:, ], y_pred_GBRT[in_out_1990:, ])
# RF
MSFE_RF_1990 = mean_squared_error(y_pred_RF[in_out_1990:, ], actual[in_out_1990:, ])
IS_R_RF_1990 = 1 - MSFE_RF_1990 / MSFE_HA
MSFE_adjusted_RF_1990, p_RF_1990 = CW_test(actual[in_out_1990:, ], y_pred_HA[in_out_1990:, ], y_pred_RF[in_out_1990:, ])
# NN3
MSFE_NN3_1990 = mean_squared_error(y_pred_NN3[in_out_1990:, ], actual[in_out_1990:, ])
IS_R_NN3_1990 = 1 - MSFE_NN3_1990 / MSFE_HA
MSFE_adjusted_NN3_1990, p_NN3_1990 = CW_test(actual[in_out_1990:, ], y_pred_HA[in_out_1990:, ], y_pred_NN3[in_out_1990:, ])
## other commonly used ML methods
# SVR
MSFE_SVR_1990 = mean_squared_error(y_pred_SVR[in_out_1990:, ], actual[in_out_1990:, ])
IS_R_SVR_1990 = 1 - MSFE_SVR_1990 / MSFE_HA
MSFE_adjusted_SVR_1990, p_SVR_1990 = CW_test(actual[in_out_1990:, ], y_pred_HA[in_out_1990:, ], y_pred_SVR[in_out_1990:, ])
# KNR
MSFE_KNR_1990 = mean_squared_error(y_pred_KNR[in_out_1990:, ], actual[in_out_1990:, ])
IS_R_KNR_1990 = 1 - MSFE_KNR_1990 / MSFE_HA
MSFE_adjusted_KNR_1990, p_KNR_1990 = CW_test(actual[in_out_1990:, ], y_pred_HA[in_out_1990:, ], y_pred_KNR[in_out_1990:, ])
# AdaBoost
MSFE_AdaBoost_1990 = mean_squared_error(y_pred_AdaBoost[in_out_1990:, ], actual[in_out_1990:, ])
IS_R_AdaBoost_1990 = 1 - MSFE_AdaBoost_1990 / MSFE_HA
MSFE_adjusted_AdaBoost_1990, p_AdaBoost_1990 = CW_test(actual[in_out_1990:, ], y_pred_HA[in_out_1990:, ], y_pred_AdaBoost[in_out_1990:, ])
# XGBoost
MSFE_XGBoost_1990 = mean_squared_error(y_pred_XGBoost[in_out_1990:, ], actual[in_out_1990:, ])
IS_R_XGBoost_1990 = 1 - MSFE_XGBoost_1990 / MSFE_HA
MSFE_adjusted_XGBoost_1990, p_XGBoost_1990 = CW_test(actual[in_out_1990:, ], y_pred_HA[in_out_1990:, ], y_pred_XGBoost[in_out_1990:, ])
# Combination
MSFE_combination_1990 = mean_squared_error(y_pred_combination[in_out_1990:, ], actual[in_out_1990:, ])
IS_R_combination_1990 = 1 - MSFE_combination_1990 / MSFE_HA
MSFE_adjusted_combination_1990, p_combination_1990 = CW_test(actual[in_out_1990:, ], y_pred_HA[in_out_1990:, ], y_pred_combination[in_out_1990:, ])

In [10]:
# results for all data set
results_in_sample_forecast1 = np.array([
    [IS_R_OLS_ALL, MSFE_adjusted_OLS_ALL, p_OLS_ALL, IS_R_OLS_1957, MSFE_adjusted_OLS_1957, p_OLS_1957, 
     IS_R_OLS_1990, MSFE_adjusted_OLS_1990, p_OLS_1990],
    [IS_R_PLS_ALL, MSFE_adjusted_PLS_ALL, p_PLS_ALL, IS_R_PLS_1957, MSFE_adjusted_PLS_1957, p_PLS_1957,
     IS_R_PLS_1990, MSFE_adjusted_PLS_1990, p_PLS_1990],
    [IS_R_PCR_ALL, MSFE_adjusted_PCR_ALL, p_PCR_ALL, IS_R_PCR_1957, MSFE_adjusted_PCR_1957, p_PCR_1957,
     IS_R_PCR_1990, MSFE_adjusted_PCR_1990, p_PCR_1990],
    [IS_R_LASSO_ALL, MSFE_adjusted_LASSO_ALL, p_LASSO_ALL, IS_R_LASSO_1957, MSFE_adjusted_LASSO_1957, p_LASSO_1957,
     IS_R_LASSO_1990, MSFE_adjusted_LASSO_1990, p_LASSO_1990],
    [IS_R_ENet_ALL, MSFE_adjusted_ENet_ALL, p_ENet_ALL, IS_R_ENet_1957, MSFE_adjusted_ENet_1957, p_ENet_1957,
     IS_R_ENet_1990, MSFE_adjusted_ENet_1990, p_ENet_1990],
    [IS_R_GBRT_ALL, MSFE_adjusted_GBRT_ALL, p_GBRT_ALL, IS_R_GBRT_1957, MSFE_adjusted_GBRT_1957, p_GBRT_1957,
     IS_R_GBRT_1990, MSFE_adjusted_GBRT_1990, p_GBRT_1990],
    [IS_R_RF_ALL, MSFE_adjusted_RF_ALL, p_RF_ALL, IS_R_RF_1957, MSFE_adjusted_RF_1957, p_RF_1957,
     IS_R_RF_1990, MSFE_adjusted_RF_1990, p_RF_1990],
    [IS_R_NN3_ALL, MSFE_adjusted_NN3_ALL, p_NN3_ALL, IS_R_NN3_1957, MSFE_adjusted_NN3_1957, p_NN3_1957,
     IS_R_NN3_1990, MSFE_adjusted_NN3_1990, p_NN3_1990]
])
results_in_sample_forecast1 = pd.DataFrame(results_in_sample_forecast1)
results_in_sample_forecast1.columns = ['All-' + s for s in ['R2', 'CW_stat', 'pValue']] +  \
    ['1957-' + s for s in ['R2', 'CW_stat', 'pValue']] + \
    ['1990-' + s for s in ['R2', 'CW_stat', 'pValue']]

results_in_sample_forecast1.insert(0, "Forecasting models",  ["OLS", "PLS", "PCR", "LASSO", "ENet",
                                                              "GBRT", "RF", "NN3"])
results_in_sample_forecast1.to_csv("results_in_sample_forecast1.csv", index=False)
results_in_sample_forecast1

,Forecasting models,All-R2,All-CW_stat,All-pValue,1957-R2,1957-CW_stat,1957-pValue,1990-R2,1990-CW_stat,1990-pValue
0,OLS,0.055098,4.389696,5.675465e-06,0.040749,4.501040,3.381089e-06,0.019757,2.393902,8.335108e-03
1,PLS,0.033404,3.276532,5.254517e-04,0.024160,3.613681,1.509403e-04,-0.003130,1.231696,1.090312e-01
2,PCR,0.018461,2.258524,1.195649e-02,0.007727,2.142736,1.606717e-02,-0.003120,0.641321,2.606572e-01
3,LASSO,0.032120,4.105958,2.013216e-05,0.037853,3.770126,8.158248e-05,0.021094,1.662091,4.824729e-02
4,ENet,0.032084,4.101678,2.050827e-05,0.037802,3.768634,8.207158e-05,0.021087,1.661705,4.828600e-02
5,GBRT,0.538454,5.694044,6.203272e-09,0.376251,6.460093,5.231926e-11,0.394570,5.226563,8.634492e-08
6,RF,0.848473,10.635833,0.000000e+00,0.845093,13.051034,0.000000e+00,0.832529,10.368308,0.000000e+00
7,NN3,0.019744,3.955070,3.825619e-05,0.003377,1.891119,2.930424e-02,0.033137,2.208661,1.359911e-02


In [11]:
results_in_sample_forecast2 = np.array([
    [IS_R_SVR_ALL, MSFE_adjusted_SVR_ALL, p_SVR_ALL, IS_R_SVR_1957, MSFE_adjusted_SVR_1957, p_SVR_1957, 
     IS_R_SVR_1990, MSFE_adjusted_SVR_1990, p_SVR_1990],
    [IS_R_KNR_ALL, MSFE_adjusted_KNR_ALL, p_KNR_ALL, IS_R_KNR_1957, MSFE_adjusted_KNR_1957, p_KNR_1957,
     IS_R_KNR_1990, MSFE_adjusted_KNR_1990, p_KNR_1990],
    [IS_R_AdaBoost_ALL, MSFE_adjusted_AdaBoost_ALL, p_AdaBoost_ALL, IS_R_AdaBoost_1957, MSFE_adjusted_AdaBoost_1957, p_AdaBoost_1957,
     IS_R_AdaBoost_1990, MSFE_adjusted_AdaBoost_1990, p_AdaBoost_1990],
    [IS_R_XGBoost_ALL, MSFE_adjusted_XGBoost_ALL, p_XGBoost_ALL, IS_R_XGBoost_1957, MSFE_adjusted_XGBoost_1957, p_XGBoost_1957,
     IS_R_XGBoost_1990, MSFE_adjusted_XGBoost_1990, p_XGBoost_1990],
    [IS_R_combination_ALL, MSFE_adjusted_combination_ALL, p_combination_ALL, IS_R_combination_1957, MSFE_adjusted_combination_1957, p_combination_1957,
     IS_R_combination_1990, MSFE_adjusted_combination_1990, p_combination_1990]
])
results_in_sample_forecast2 = pd.DataFrame(results_in_sample_forecast2)
results_in_sample_forecast2.columns = ['All-' + s for s in ['R2', 'CW_stat', 'pValue']] +  \
    ['1957-' + s for s in ['R2', 'CW_stat', 'pValue']] + \
    ['1990-' + s for s in ['R2', 'CW_stat', 'pValue']]
results_in_sample_forecast2.insert(0, "Forecasting models",
                                   ["SVR", "KNR", "AdaBoost", "XGBoost", "Combination"])
results_in_sample_forecast2.to_csv("results_in_sample_forecast2.csv", index=False)
results_in_sample_forecast2

,Forecasting models,All-R2,All-CW_stat,All-pValue,1957-R2,1957-CW_stat,1957-pValue,1990-R2,1990-CW_stat,1990-pValue
0,SVR,0.011637,3.371052,3.744084e-04,-0.206786,2.646218,0.004070,-0.220495,1.571677,5.801280e-02
1,KNR,0.235314,7.680542,7.882583e-15,0.199688,8.420183,0.000000,0.198059,5.539477,1.516884e-08
2,AdaBoost,0.187070,3.415776,3.180025e-04,-0.160667,1.945482,0.025859,-0.279358,1.044129,1.482129e-01
3,XGBoost,0.992909,10.257065,0.000000e+00,0.988749,12.896891,0.000000,0.990155,9.991474,0.000000e+00
4,Combination,0.384925,7.424444,5.662137e-14,0.326639,9.478794,0.000000,0.316579,7.123857,5.246914e-13
